In [5]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, csr_matrix, save_npz
from sklearn.preprocessing import normalize

In [7]:
tfidf_matrix = load_npz("../data/output/TFIDF.npz")

doc_index = pd.read_csv(
    "../data/output/tfidf_doc_index.csv",
    index_col=0
).squeeze().to_dict()

term_index = pd.read_csv(
    "../data/output/tfidf_term_index.csv",
    index_col=0
).squeeze().to_dict()

print("TFIDF shape:", tfidf_matrix.shape)

TFIDF shape: (31102, 12878)


In [8]:
k = 1000

# Sum TFIDF across documents
term_scores = np.array(tfidf_matrix.sum(axis=0)).flatten()

top_term_indices = np.argsort(term_scores)[-k:]

reduced_tfidf = tfidf_matrix[:, top_term_indices]

print("Reduced TFIDF shape:", reduced_tfidf.shape)

Reduced TFIDF shape: (31102, 1000)


In [9]:
tfidf_l2 = normalize(reduced_tfidf, norm="l2", axis=1)

print("L2 Normalized shape:", tfidf_l2.shape)

L2 Normalized shape: (31102, 1000)


In [10]:
row_norms = np.linalg.norm(tfidf_l2.toarray(), axis=1)

print("Min row norm:", row_norms.min())
print("Max row norm:", row_norms.max())

Min row norm: 0.0
Max row norm: 1.0000000000000002


In [17]:
delimiter = "|"
print("Delimiter:", delimiter)

print("Number of features (i.e. significant words):", k)

principle_text = f"""
Principle of significant word selection:

Top-k terms ranked by total TFIDF weight across all documents,
retaining the top {k} highest-scoring terms.

Significant words were selected by ranking terms according to
their total TFIDF weight across all documents and retaining
the top k terms, prioritizing features that are both informative
and substantively present in the corpus.
"""

print(principle_text)

Delimiter: |
Number of features (i.e. significant words): 1000

Principle of significant word selection:

Top-k terms ranked by total TFIDF weight across all documents,
retaining the top 1000 highest-scoring terms.

Significant words were selected by ranking terms according to
their total TFIDF weight across all documents and retaining
the top k terms, prioritizing features that are both informative
and substantively present in the corpus.



In [18]:
os.makedirs("../data/output", exist_ok=True)

save_npz("../data/output/TFIDF_L2_REDUCED.npz", tfidf_l2)

print("Reduced L2 TFIDF saved.")

Reduced L2 TFIDF saved.
